In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('/Users/yudhapratama/Documents/Bahan Kuliah/UrbanAnalytics/updated.csv')
df

,Unnamed: 0,id,year,water,morbid,area,pop,pad,daerah,tingkat,logpop,logpad,logarea,status,kabko,lat_x,long_x,lat_y,long_y
0,0,ID.BE.BS,1996,12.99,17.11,5949.00,315900.0,9.302870e+08,Bengkulu Selatan,Kab.,12.663181,20.651004,8.690978,Kota Sedang,KAB. BENGKULU SELATAN,-4.3248,103.0357,-4.3248,103.0357
1,1,ID.BE.BS,1997,14.60,17.54,5949.00,317600.0,1.448954e+09,Bengkulu Selatan,Kab.,12.668548,21.094108,8.690978,Kota Sedang,KAB. BENGKULU SELATAN,-4.3248,103.0357,-4.3248,103.0357
2,2,ID.BE.BS,1998,13.99,23.03,5949.00,318900.0,1.608408e+09,Bengkulu Selatan,Kab.,12.672633,21.198511,8.690978,Kota Sedang,KAB. BENGKULU SELATAN,-4.3248,103.0357,-4.3248,103.0357
3,3,ID.BE.BS,1999,8.20,15.82,5949.00,319800.0,1.686854e+09,Bengkulu Selatan,Kab.,12.675451,21.246131,8.690978,Kota Sedang,KAB. BENGKULU SELATAN,-4.3248,103.0357,-4.3248,103.0357
4,4,ID.BE.BS,2000,9.01,20.42,5949.00,370678.0,6.493360e+08,Bengkulu Selatan,Kab.,12.823089,20.291461,8.690978,Kota Sedang,KAB. BENGKULU SELATAN,-4.3248,103.0357,-4.3248,103.0357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4600,4600,ID.LA.WK,2016,38.02,25.04,3921.63,437530.0,4.434695e+10,Way Kanan,Kab.,12.988901,24.515310,8.274263,Kota Sedang,KAB. WAY KANAN,-4.4964,104.5655,-4.4964,104.5655
4601,4601,ID.LA.WK,2017,17.79,30.87,3921.63,441922.0,4.955128e+10,Way Kanan,Kab.,12.998889,24.626274,8.274263,Kota Sedang,KAB. WAY KANAN,-4.4964,104.5655,-4.4964,104.5655
4602,4602,ID.LA.WK,2018,27.24,34.80,3921.63,446113.0,5.676499e+10,Way Kanan,Kab.,13.008328,24.762186,8.274263,Kota Sedang,KAB. WAY KANAN,-4.4964,104.5655,-4.4964,104.5655
4603,4603,ID.LA.WK,2019,34.52,31.26,3921.63,450109.0,6.235356e+10,Way Kanan,Kab.,13.017245,24.856087,8.274263,Kota Sedang,KAB. WAY KANAN,-4.4964,104.5655,-4.4964,104.5655


In [3]:
X = df[['logpop','logarea','logpad','year','morbid','daerah','tingkat','status']]
y = df['water']

In [4]:
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, TimeSeriesSplit

X_train, X_test, y_train, y_test= train_test_split(
    X,
    y,  
    test_size=0.2,
    shuffle=True)

# Model Prep

In [7]:
#Transform
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import category_encoders as ce
from sklearn.preprocessing import RobustScaler

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost.sklearn import XGBRegressor

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, GridSearchCV, KFold

from sklearn.pipeline import Pipeline

In [6]:
conda install category_encoders

Solving environment: done

## Package Plan ##

  environment location: /Users/yudhapratama/anaconda3

  added / updated specs:
    - category_encoders


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    category_encoders-2.6.0    |  py310hca03da5_0         110 KB
    ------------------------------------------------------------
                                           Total:         110 KB

The following NEW packages will be INSTALLED:

  category_encoders  pkgs/main/osx-arm64::category_encoders-2.6.0-py310hca03da5_0 



                                                                                
Preparing transaction: done
Verifying transaction: done
Executing transaction: done

Note: you may need to restart the kernel to use updated packages.


In [8]:
transformer = ColumnTransformer(
    transformers=[
        ('kategorikal', OneHotEncoder(drop='first'), ['year','status','tingkat']),
        ('Binary',ce.BinaryEncoder(),['daerah'])
    ],remainder='passthrough'
)

scaler = RobustScaler()

In [9]:
# Define algoritma yang digunakan
lr = LinearRegression()
knn = KNeighborsRegressor()
dt = DecisionTreeRegressor(random_state=1)
rf = RandomForestRegressor(random_state=1)
xgb = XGBRegressor(random_state=1)

models = [lr,knn,dt,rf,xgb]
          
score_rmse = []
nilai_mean_rmse = []
nilai_std_rmse = []

score_mae = []
nilai_mean_mae = []
nilai_std_mae = []

score_mape = []
nilai_mean_mape = []
nilai_std_mape = []

In [10]:
# Mencari algoritma terbaik berdasarkan tiap metrics
for i in models:
    
    crossval = KFold(n_splits=5, shuffle=True, random_state=1)

    estimator = Pipeline([
        ('preprocessing', transformer),
        ('scaler', scaler),
        ('model', i)
    ])

    # RMSE
    model_cv_rmse = cross_val_score(
        estimator, 
        X_train, 
        y_train, 
        cv=crossval, 
        scoring='neg_root_mean_squared_error', 
        error_score='raise'
        )

    print(model_cv_rmse, i)

    score_rmse.append(model_cv_rmse)
    nilai_mean_rmse.append(model_cv_rmse.mean())
    nilai_std_rmse.append(model_cv_rmse.std())

    # MAE
    model_cv_mae = cross_val_score(
        estimator, 
        X_train, 
        y_train, 
        cv=crossval, 
        scoring='neg_mean_absolute_error', 
        error_score='raise'
        )

    print(model_cv_mae, i)

    score_mae.append(model_cv_mae)
    nilai_mean_mae.append(model_cv_mae.mean())
    nilai_std_mae.append(model_cv_mae.std())

    # MAPE
    model_cv_mape = cross_val_score(
        estimator, 
        X_train, 
        y_train, 
        cv=crossval, 
        scoring='neg_mean_absolute_percentage_error', 
        error_score='raise'
        )

    print(model_cv_mape, i)

    score_mape.append(model_cv_mape)
    nilai_mean_mape.append(model_cv_mape.mean())
    nilai_std_mape.append(model_cv_mape.std())

[-14.13711335 -14.08054928 -14.16954114 -14.53840916 -13.72124097] LinearRegression()
[-11.25875077 -11.10428926 -11.22383587 -11.40903172 -10.87115155] LinearRegression()
[-0.37271206 -0.33701109 -0.32978152 -0.38033596 -0.27685317] LinearRegression()
[-10.26054992  -9.36431648 -10.21631     -9.85093329  -9.96764735] KNeighborsRegressor()
[-7.80885753 -7.36229037 -7.60240163 -7.53319674 -7.52523913] KNeighborsRegressor()
[-0.24252233 -0.20521693 -0.22368191 -0.20782282 -0.20429376] KNeighborsRegressor()
[-12.29880646 -14.13284517 -12.25716816 -13.97841777 -13.15728981] DecisionTreeRegressor(random_state=1)
[ -8.71233379 -10.00081411  -8.48565807  -9.8465943   -9.2578125 ] DecisionTreeRegressor(random_state=1)
[-0.24534871 -0.29068437 -0.21293636 -0.24650493 -0.22981115] DecisionTreeRegressor(random_state=1)
[-9.14342918 -9.68963648 -9.75803978 -9.79617039 -9.99072513] RandomForestRegressor(random_state=1)
[-7.13605156 -7.29073365 -7.36427585 -7.45712429 -7.29362908] RandomForestRegres

In [11]:
# Evaluasi hasil dari 5 kandidat algoritma yang digunakan
pd.DataFrame({
    'Model': ['Linear Regression', 'KNN Regressor', 'DecisionTree Regressor', 'RandomForest Regressor', 'XGBoost Regressor'],
    'Mean_RMSE': nilai_mean_rmse,
    'Std_RMSE': nilai_std_rmse,
    'Mean_MAE': nilai_mean_mae,
    'Std_MAE': nilai_std_mae,
    'Mean_MAPE': nilai_mean_mape,
    'Std_MAPE': nilai_std_mape
})

,Model,Mean_RMSE,Std_RMSE,Mean_MAE,Std_MAE,Mean_MAPE,Std_MAPE
0,Linear Regression,-14.129371,0.259977,-11.173412,0.179712,-0.339339,0.036867
1,KNN Regressor,-9.931951,0.322144,-7.566397,0.144595,-0.216708,0.014698
2,DecisionTree Regressor,-13.164905,0.796603,-9.260643,0.598739,-0.245057,0.025877
3,RandomForest Regressor,-9.675600,0.284268,-7.308363,0.105333,-0.212592,0.012638
4,XGBoost Regressor,-9.148597,0.339671,-6.978229,0.193156,-0.192898,0.013473


In [12]:
# Benchmark 2 model terbaik
models = {
    'XGB': XGBRegressor(random_state=1),
    'RandomForest': RandomForestRegressor(random_state=1)
}

score_rmse = []
score_mae = []
score_mape = []

# Prediksi pada test set
for i in models:

    model = Pipeline([
        ('preprocessing', transformer),
        ('scaler', scaler),
        ('model', models[i])
        ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    score_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))
    score_mae.append(mean_absolute_error(y_test, y_pred))
    score_mape.append(mean_absolute_percentage_error(y_test, y_pred))

score_before_tuning = pd.DataFrame({'RMSE': score_rmse, 'MAE': score_mae, 'MAPE': score_mape}, index=models.keys())
score_before_tuning

,RMSE,MAE,MAPE
XGB,9.223428,6.998126,0.196543
RandomForest,9.617037,7.294476,0.216790


# Hyperparameter Tuning

In [13]:
# xgb
max_depth = list(np.arange(3, 11)) # Kedalaman pohon
learning_rate = list(np.arange(1, 4)/100) # Learning rate
n_estimators = list(np.arange(100, 1001)) # Jumlah pohon
subsample = list(np.arange(6, 11)/10) # Jumlah baris tiap pohon (% dari total baris train set)
gamma = list(np.arange(1, 6)/10) # Semakin besar nilainya, semakin konservatif/simpel modelnya
colsample_bytree = list(np.arange(5, 11)/10) # Jumlah feature yang digunakan untuk tiap pohon (% dari total kolom train set)
reg_alpha = list(np.arange(5, 50)/100) # Semakin besar nilainya, semakin konservatif/simpel modelnya

min_child_weight = list(np.arange(1, 11))

# Hyperparam space XGboost
hyperparam_space_xgb = {
    'model__max_depth': max_depth, 
    'model__learning_rate': learning_rate,
    'model__n_estimators': n_estimators,
    'model__subsample': subsample,
    'model__gamma': gamma,
    'model__colsample_bytree': colsample_bytree,
    'model__reg_alpha': reg_alpha,
    'model__eta': eta,
}

In [16]:
# Benchmark model dengan hyperparameter tuning
xgb = XGBRegressor(random_state=2023)

# Membuat algorithm chains
estimator_xgb = Pipeline([
        ('preprocessing', transformer),
        ('scaler',scaler),
        ('model', xgb)
        ])

crossval = crossval

# Hyperparameter tuning
random_xgb = RandomizedSearchCV(
    estimator_xgb, 
    param_distributions = hyperparam_space_xgb,
    n_iter = 50,
    cv = crossval, 
    scoring = ['neg_root_mean_squared_error', 'neg_mean_absolute_percentage_error'], 
    n_jobs = -1,
    refit = 'neg_root_mean_squared_error', # Hanya bisa memilih salah stau metric untuk optimisasi
    random_state = 2022  
)

In [17]:
random_xgb.fit(X_train, y_train)

RandomizedSearchCV(cv=KFold(n_splits=5, random_state=1, shuffle=True),
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(remainder='passthrough',
                                                                transformers=[('kategorikal',
                                                                               OneHotEncoder(drop='first'),
                                                                               ['year',
                                                                                'status',
                                                                                'tingkat']),
                                                                              ('Binary',
                                                                               BinaryEncoder(),
                                                                               ['daerah'])])),
                                             ('scaler', RobustScaler()),
                                             ('model',
                                              XGBRegressor(base_score=None,
                                                           booste...
                                                                127, 128, 129, ...],
                                        'model__reg_alpha': [0.05, 0.06, 0.07,
                                                             0.08, 0.09, 0.1,
                                                             0.11, 0.12, 0.13,
                                                             0.14, 0.15, 0.16,
                                                             0.17, 0.18, 0.19,
                                                             0.2, 0.21, 0.22,
                                                             0.23, 0.24, 0.25,
                                                             0.26, 0.27, 0.28,
                                                             0.29, 0.3, 0.31,
                                                             0.32, 0.33, 0.34, ...],
                                        'model__subsample': [0.6, 0.7, 0.8, 0.9,
                                                             1.0]},
                   random_state=2022, refit='neg_root_mean_squared_error',
                   scoring=['neg_root_mean_squared_error',
                            'neg_mean_absolute_percentage_error'])

In [18]:
# Cek skor dan params terbaik
print('XGBoost')
print('Best_score:', random_xgb.best_score_)
print('Best_params:', random_xgb.best_params_)

XGBoost
Best_score: -8.285116553495689
Best_params: {'model__subsample': 0.9, 'model__reg_alpha': 0.1, 'model__n_estimators': 837, 'model__max_depth': 10, 'model__learning_rate': 0.02, 'model__gamma': 0.2, 'model__eta': 0.1, 'model__colsample_bytree': 0.7}


In [19]:
# Model XGBoost
model = {'XGB': XGBRegressor(random_state=2023)}

# Define model terhadap estimator terbaik
tuned_model = random_xgb.best_estimator_

# Fitting model
tuned_model.fit(X_train, y_train)

# Predict test set
y_pred_tuned = tuned_model.predict(X_test)

# Simpan nilai metrics RMSE, MAE setelah tuning
rmse_xgb_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
mape_xgb_tuned = mean_absolute_percentage_error(y_test, y_pred_tuned)

score_after_tuning = pd.DataFrame({'RMSE': rmse_xgb_tuned, 'MAPE': mape_xgb_tuned}, index=model.keys())
score_after_tuning

,RMSE,MAPE
XGB,8.447658,0.184933


In [ ]:
# Performance Comparison

In [ ]:
## FTXG

In [20]:
pd.DataFrame(score_before_tuning.iloc[0])

,XGB
RMSE,9.223428
MAE,6.998126
MAPE,0.196543


In [21]:
score_after_tuning.T

,XGB
RMSE,8.447658
MAPE,0.184933


# Export Model

In [ ]:
import pickle

estimator = Pipeline([
        ('preprocessing',transformer),
        ('scaler', scaler),
        ('model',XGBRegressor())
    ])
estimator.fit(X_train,y_train)

In [ ]:
pickle.dump(estimator, open('Model_AirBersih_.pkl', 'wb'))

In [31]:
import pickle
from time import time

# set filename
milliseconds = int(time() * 1000)
filename = f'model_{milliseconds}.sav'

#save the model to disk
pickle.dump(tuned_model, open(filename, 'wb'))


In [32]:
X_test

,logpop,logarea,logpad,year,morbid,daerah,tingkat,status
219,12.884614,6.373797,24.707652,2011,40.20,Kulon Progo,Kab.,Kota Sedang
2794,13.953542,7.436152,26.124678,2016,24.20,Probolinggo,Kab.,Metropolitan
1687,12.477054,3.812203,22.714883,2001,29.34,Pekalongan,Kota,Kota Sedang
60,12.490052,4.973418,23.539280,2006,24.57,Bengkulu,Kota,Kota Sedang
501,12.598486,8.468387,22.665325,2006,25.68,Muaro Jambi,Kab.,Kota Sedang
...,...,...,...,...,...,...,...,...
439,13.263979,4.639958,26.298933,2015,22.25,Jambi,Kota,Kota Besar
512,12.950813,8.580356,25.079036,2017,15.62,Muaro Jambi,Kab.,Kota Sedang
296,11.972456,7.327702,24.995578,2017,32.17,Boalemo,Kab.,Kota Sedang
2058,13.443142,6.892642,22.196415,1997,29.08,Wonosobo,Kab.,Kota Besar


# Mencoba Model

In [40]:
#create new DataFrame
df_new = pd.DataFrame({'logpop': [12.88,13],
                       'logarea': [6,7],
                       'logpad' : [26,18],
                       'year': [2012,2017],
                       'morbid': [40,30],
                       'daerah' : ['Kulon Progo','Pekalongan'],
                       'tingkat' : ['Kab.','Kota'],
                       'status' : ['Kota Besar','Kota Sedang']
                      })
df_new

,logpop,logarea,logpad,year,morbid,daerah,tingkat,status
0,12.88,6,26,2012,40,Kulon Progo,Kab.,Kota Besar
1,13.00,7,18,2017,30,Pekalongan,Kota,Kota Sedang


In [41]:
tuned_model.predict(df_new)

array([74.730385, 44.922523], dtype=float32)